# 1 DATA PREPARATION

**Project:** Irish Company Survival: Registrations, Cohort Survival and Prediction

**Author:** Temuulen Bulgan

### Purpose of this notebook
This notebook loads the raw company register, checks its quality and turns it into one clean table of companies that the rest of the project uses for analysis, SQL and the Power BI dashboard.

### Data sources
| Dataset | Source | Content |
|---|---|---|
| Company Records (`companies.csv.zip`) | Companies Registration Office (CRO) | One row per company registered in Ireland, including dissolved companies: registration date, status, dissolution date, company type, sector code and address |

### Contents
1. Company Records: inspection
2. Company Records: cleaning
3. Saving the clean data

### Output
This notebook saves the cleaned data to `data/processed/companies_clean.csv` (one row per company).

## Setup

In [1]:
import pandas as pd
from pathlib import Path

# Show all columns when displaying a DataFrame
pd.set_option("display.max_columns", None)

## 1. Company Records: inspection

**Source:** Companies Registration Office (CRO), dataset "Company Records". Downloaded from opendata.cro.ie in October 2026. Licence: CC BY 4.0. Contains Irish Public Sector Data licensed under a Creative Commons Attribution 4.0 International (CC BY 4.0) licence.

**Structure according to the source:** one row per company registered in Ireland, including dissolved companies, with registration date, status, dissolution date, company type, sector codes and address.

In this section I inspect the raw file before cleaning it.

### 1.1 Inspection

#### Load the file and check its size

In [2]:
# Load the raw file
companies = pd.read_csv("../data/raw/companies.csv.zip")

# Number of rows and columns
companies.shape

(827902, 21)

The file has 827,902 rows (one per company) and 21 columns.

#### First and last rows

In [3]:
# First and last 3 rows: format of the data, and where it starts and ends
display(companies.head(3))
display(companies.tail(3))

,company_num,company_name,company_status_code,company_status,company_type_code,company_type,company_reg_date,last_ar_date,company_address_1,company_address_2,company_address_3,company_address_4,comp_dissolved_date,nard,last_accounts_date,company_status_date,nace_v2_code,eircode,company_name_eff_date,company_type_eff_date,princ_object_code
0,340217,STORM ENTERPRISES LIMITED,1151,Normal,1153,LTD - Private Company Limited by Shares,2001-03-13,2025-09-30,C/O Storm Technology,The Capel Building,Marys Abbey,",DUBLIN",NaN,2026-09-30,2024-12-31,NaN,NaN,D07 N977,2015-11-05,2016-12-01,72.20
1,340235,L.M.G. DEVELOPMENTS LIMITED,1158,Dissolved,1153,LTD - Private Company Limited by Shares,2001-03-13,2016-09-13,7 GLEBE CREST,DONEGAL TOWN,"CO. DONEGAL,DONEGAL","CO. DONEGAL, DONEGAL, IRELAND",2017-11-29,2017-09-13,2016-03-31,2017-11-29,NaN,F94H4X5,2001-03-13,2016-12-01,45.21
2,340241,M V FOODS LIMITED,1151,Normal,1153,LTD - Private Company Limited by Shares,2001-03-13,2025-12-31,UNIT 1 CLANBRASSIL CENTRE,CLANBRASSIL STREET,DUNDALK CO.LOUTH.,"DUNDALK,LOUTH",NaN,2026-12-31,2025-03-31,NaN,NaN,NaN,2001-03-13,2016-12-01,52.22


,company_num,company_name,company_status_code,company_status,company_type_code,company_type,company_reg_date,last_ar_date,company_address_1,company_address_2,company_address_3,company_address_4,comp_dissolved_date,nard,last_accounts_date,company_status_date,nace_v2_code,eircode,company_name_eff_date,company_type_eff_date,princ_object_code
827899,829192,CK CASTLE PROPERTY SERVICES LIMITED,1151,Normal,1153,LTD - Private Company Limited by Shares,2026-10-27,NaN,82 PARK STREET,NaN,NaN,"DUNDALK,LOUTH",NaN,2027-04-27,NaN,NaN,5630.0,A91EC85,2026-10-27,2026-10-27,NaN
827900,829212,ANGEL GLOBAL VOYAGER LIMITED,1151,Normal,1153,LTD - Private Company Limited by Shares,2026-10-27,NaN,THE BLACK CHURCH,ST. MARY'S PLACE,NaN,DUBLIN 7,NaN,2027-04-27,NaN,NaN,5829.0,D07P4AX,2026-10-27,2026-10-27,NaN
827901,829189,SLM DENTAL CARE LIMITED,1151,Normal,1153,LTD - Private Company Limited by Shares,2026-10-27,NaN,Birch Lane,Iskaheen,"Muff,DONEGAL",NaN,NaN,2027-04-27,NaN,NaN,8623.0,F93 RD83,2026-10-27,2026-10-27,NaN


First look: there is one row per company. The file does not start with the oldest companies: the first rows were registered on 13 March 2001 and the last rows on 27 October 2026. The company numbers are not in strict order either. The 2001 rows have a sector code in `princ_object_code` but not in `nace_v2_code`, and the 2026 rows have the opposite. `nace_v2_code` is shown with a decimal point (for example `5630.0`). Some text has extra spaces or mixed formats, for example the addresses (`,DUBLIN`, `CO. DONEGAL,DONEGAL`).

#### Column types and missing counts

In [4]:
# Column names, data types and non-missing counts
companies.info()

<class 'pandas.DataFrame'>
RangeIndex: 827902 entries, 0 to 827901
Data columns (total 21 columns):
 #   Column                 Non-Null Count   Dtype  
---  ------                 --------------   -----  
 0   company_num            827902 non-null  int64  
 1   company_name           827902 non-null  str    
 2   company_status_code    827902 non-null  int64  
 3   company_status         827902 non-null  str    
 4   company_type_code      827902 non-null  int64  
 5   company_type           827902 non-null  str    
 6   company_reg_date       811710 non-null  str    
 7   last_ar_date           610066 non-null  str    
 8   company_address_1      827902 non-null  str    
 9   company_address_2      766947 non-null  str    
 10  company_address_3      572718 non-null  str    
 11  company_address_4      802369 non-null  str    
 12  comp_dissolved_date    475006 non-null  str    
 13  nard                   802906 non-null  str    
 14  last_accounts_date     501102 non-null  str    

The identifier, name, status and type columns are complete (827,902 non-null). `company_reg_date` has fewer values (811,710), so some companies have no registration date. This matters because every survival measure starts from this date. All date columns are stored as text (`str`), so they need converting to dates. `nace_v2_code` is stored as a number (`float64`), which explains the `.0` seen earlier. It should be text, because it is a code, not an amount. Both sector code columns and `comp_dissolved_date` are only partly filled. The next check counts the missing values in each column.

#### Missing values per column

In [5]:
# Number of missing values in each column
missing_count = companies.isna().sum()

# Missing values as a percentage of all rows
missing_pct = (missing_count / len(companies) * 100).round(1)

# Put both in one table, largest first
missing = pd.DataFrame({"missing": missing_count, "missing_pct": missing_pct})
missing.sort_values("missing", ascending=False)

,missing,missing_pct
nace_v2_code,561909,67.9
eircode,533605,64.5
princ_object_code,369827,44.7
comp_dissolved_date,352896,42.6
last_accounts_date,326800,39.5
company_status_date,320925,38.8
company_address_3,255184,30.8
last_ar_date,217836,26.3
company_address_2,60955,7.4
company_address_4,25533,3.1
